In [1]:
import sys
import itertools
sys.path.append('../')
import random
from core import LADTransferTreeBoost, LSTransferTreeBoost, MTransferTreeBoost
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb #baseline
from utils import *
from baselines import *
from friedman1 import *

In [2]:
test_size = 1000
val_size = 1000

target_instances_list = [100,200,300]
source_instances = 1000 #we use a fixed number of source instances

d_list = [1,2,3,4,5,6,7,8,9,10]
seed_list = [1,2,3,4,5]

In [3]:
#ablation study for LSTransferTreeBoost with Gaussian errors, with gaussian source domain errors
ablation_transfer_normal_normal = pd.DataFrame(columns = ['seed', 'target_instances', 'd', 'method', 'v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 
                                                          'val_rmse', 'val_mae', 'rmse', 'mae'])
v_list = [0.05, 0.1]
source_tree_size_list = [1,2]
target_tree_size_list = [1,2]
k_list = [0.01, 0.05]
m_0_list = [0.5, 0.9]


# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    v_list,
    source_tree_size_list,
    target_tree_size_list,
    k_list,
    m_0_list
))

for seed in seed_list:
    for target_instances in target_instances_list:
    
        X_target_test, y_target_test = friedman1(n_samples=test_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #do NOT add noise to test set!!!!
        X_target_val, y_target_val = friedman1(n_samples=val_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed + 10) #do NOT add noise to test set!!!!
        X_target_train, y_target_train = friedman1(n_samples=target_instances, add_noise = True, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #add noise to train set
        for d in d_list:
            X_source_train, y_source_train = friedman1_altered(n_samples=1000, add_noise = True, noise_distribution = 'gaussian',
                                                            n_features=10, d=d, shift_seed=seed, random_seed = seed) #also add noise to source (only train here)
            for config in param_grid:
                v, source_tree_size, target_tree_size, k, m_0 = config


                #Test for all methods!!!!

                method = f'LSTransferTreeBoost'
                fiter = LSTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                                        target_tree_size=target_tree_size, k=k, m_0=m_0)
                fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves=False)
                rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
                val_rmse = fiter.evaluate(X_target_val, y_target_val, metric = 'rmse')
                mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
                val_mae = fiter.evaluate(X_target_val, y_target_val, metric = 'mae')
                ablation_transfer_normal_normal.loc[len(ablation_transfer_normal_normal)] = [seed, target_instances, d, method, v, source_tree_size, target_tree_size, k, m_0, val_rmse, val_mae, rmse, mae]
                ablation_transfer_normal_normal.to_csv(f'results/LSTransferTreeBoost_ablation_friedman.csv')





                            
        



KeyboardInterrupt: 

In [ ]:
#ablation study for transfertreeboost Gaussian errors, with gaussian source domain errors
ablation_transfer_normal_normal = pd.DataFrame(columns = ['seed', 'target_instances', 'd', 'method',
                                   'v', 'target_tree_size', 'val_rmse', 'val_mae', 'rmse', 'mae'])

v_list = [0.01, 0.02, 0.05, 0.1, 0.15]
target_tree_size_list = [1,2,3,4]

# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    v_list,
    target_tree_size_list
))



for seed in seed_list:
    for target_instances in target_instances_list:
    
        X_target_test, y_target_test = friedman1(n_samples=test_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #do NOT add noise to test set!!!!
        X_target_val, y_target_val = friedman1(n_samples=val_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed + 10) #do NOT add noise to test set!!!!
        X_target_train, y_target_train = friedman1(n_samples=target_instances, add_noise = True, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #add noise to train set
        ones = np.ones((len(X_target_train), 1))
        X_target_train_with_dummy = np.hstack((X_target_train, ones))
        ones = np.ones((len(X_target_val), 1))
        X_target_val_with_dummy = np.hstack((X_target_val, ones))
        ones = np.ones((len(X_target_test), 1))
        X_target_test_with_dummy= np.hstack((X_target_test, ones))
        for d in d_list:
            X_source_train, y_source_train = friedman1_altered(n_samples=1000, add_noise = True, noise_distribution = 'gaussian',
            
                                                        n_features=10, d=d, shift_seed=seed, random_seed = seed) #also add noise to source (only train here)
            #Create an additional set with dummy variable for source/train for naive method!           
            zeros = np.zeros((len(X_source_train), 1))
            X_source_train_with_dummy = np.hstack((X_source_train.copy(), zeros))

            for config in param_grid:
                v, target_tree_size = config


                method = 'xgboost'
                params = {
                    'objective': 'reg:squarederror',  # Regression with squared error
                    'max_depth': target_tree_size,                   # Maximum depth of a tree
                    'eta': v,                       # Learning rate
                    'eval_metric': 'rmse',           # RMSE as evaluation metric
                    }
                        
                bst = train_xgboost(X_target_train, y_target_train, X_target_val, y_target_val, boosting_rounds=1000, params=params)
                preds_val = test_xgboost(X_target_val, bst)
                val_rmse = compute_rmse(preds_val, y_target_val)
                val_mae = compute_mae(preds_val, y_target_val)
                preds = test_xgboost(X_target_test, bst)
                rmse = compute_rmse(preds, y_target_test)
                mae = compute_mae(preds, y_target_test)
                ablation_transfer_normal_normal.loc[len(ablation_transfer_normal_normal)] = [seed, target_instances, d, method, v, target_tree_size, val_rmse, val_mae,
                                                                                              rmse,mae]
                
                ablation_transfer_normal_normal.to_csv(f'results/xgboost_ablation_friedman.csv')

                method = 'xgboost_naive_transfer'
                params = {
                    'objective': 'reg:squarederror',  # Regression with squared error
                    'max_depth': target_tree_size,                   # Maximum depth of a tree
                    'eta': v,                       # Learning rate
                    'eval_metric': 'rmse',           # RMSE as evaluation metric
                    }
                X_comb = np.concatenate((X_target_train, X_source_train)) 
                y_comb = np.concatenate((y_target_train, y_source_train))       
                bst = train_xgboost(X_comb, y_comb, X_target_val, y_target_val, boosting_rounds=1000, params=params)
                preds_val = test_xgboost(X_target_val, bst)
                val_rmse = compute_rmse(preds_val, y_target_val)
                val_mae = compute_mae(preds_val, y_target_val)
                preds = test_xgboost(X_target_test, bst)
                rmse = compute_rmse(preds, y_target_test)
                mae = compute_mae(preds, y_target_test)
                ablation_transfer_normal_normal.loc[len(ablation_transfer_normal_normal)] = [seed, target_instances, d, method, v, target_tree_size, val_rmse, val_mae,
                                                                                              rmse,mae]
                
                ablation_transfer_normal_normal.to_csv(f'results/xgboost_ablation_friedman.csv')

                method = 'xgboost_naive_transfer_with_dummy'
                params = {
                    'objective': 'reg:squarederror',  # Regression with squared error
                    'max_depth': target_tree_size,                   # Maximum depth of a tree
                    'eta': v,                       # Learning rate
                    'eval_metric': 'rmse',           # RMSE as evaluation metric
                    }

                X_comb = np.concatenate((X_target_train_with_dummy, X_source_train_with_dummy)) 
                y_comb = np.concatenate((y_target_train, y_source_train))       
                bst = train_xgboost(X_comb, y_comb, X_target_val, y_target_val, boosting_rounds=1000, params=params)
                preds_val = test_xgboost(X_target_val_with_dummy, bst)
                val_rmse = compute_rmse(preds_val, y_target_val)
                val_mae = compute_mae(preds_val, y_target_val)
                preds = test_xgboost(X_target_test_with_dummy, bst)
                rmse = compute_rmse(preds, y_target_test)
                mae = compute_mae(preds, y_target_test)
                ablation_transfer_normal_normal.loc[len(ablation_transfer_normal_normal)] = [seed, target_instances, d, method, v, target_tree_size, val_rmse, val_mae,
                                                                                              rmse,mae]
                
                ablation_transfer_normal_normal.to_csv(f'results/xgboost_ablation_friedman.csv')
                                
            



In [ ]:
#also run mlp finetuning
ablation_transfer_normal_normal = pd.DataFrame(columns = ['seed', 'target_instances', 'd', 'method', 'base_lr', 'fine_tuning_lr', 'dropout_rate', 'batch_norm',
                                                          'val_rmse', 'val_mae', 'rmse', 'mae'])
fine_tuning_lrs = [1e-4, 5e-5]
base_lrs = [5e-4, 1e-4]
dropout_list = [0.0, 0.1]
include_batch_norm = [True, False]
for seed in seed_list:
    for target_instances in target_instances_list:
    
        X_target_test, y_target_test = friedman1(n_samples=test_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #do NOT add noise to test set!!!!
        X_target_val, y_target_val = friedman1(n_samples=val_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed + 10) #do NOT add noise to test set!!!!
        X_target_train, y_target_train = friedman1(n_samples=target_instances, add_noise = True, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #add noise to train set
        for d in d_list:
            X_source_train, y_source_train = friedman1_altered(n_samples=1000, add_noise = True, noise_distribution = 'gaussian',
                                                            n_features=10, d=d, shift_seed=seed, random_seed = seed) #also add noise to source (only train here)


            for base_lr in base_lrs:
                for finetuning_lr in fine_tuning_lrs:
                    for dropout_rate in dropout_list:
                        for batch_norm in include_batch_norm:

                            method = f'MLP'
                            mlp = MLP(10, 100, 100, 100, 1, dropout_rate=dropout_rate, include_batch_norm=batch_norm)
                            dataloader_train = process_dataset_for_base_network(X_source_train, y_source_train)
                            mlp, train_loss, val_loss = train_mlp_on_source(dataloader_train, mlp, epochs=1000)
                            dataloader_train, dataloader_val, dataloader_test = process_datasets_for_finetuning(X_target_train, y_target_train,
                                                        X_target_val, y_target_val, X_target_test, y_target_test, batch_size=32)
                            
                            mlp, train_loss, val_loss = finetune_mlp_on_target(dataloader_train, dataloader_val, mlp, epochs=1000, freeze_layers=None)
                            val_rmse, val_mae = test_final_mlp(dataloader_val, mlp)
                            rmse, mae = test_final_mlp(dataloader_test, mlp)
                            ablation_transfer_normal_normal.loc[len(ablation_transfer_normal_normal)] = [seed, target_instances, d, method, base_lr, finetuning_lr, dropout_rate, batch_norm, val_rmse, 
                                                                                                         val_mae, rmse, mae]
                            ablation_transfer_normal_normal.to_csv(f'results/MLP_ablation_friedman.csv')





                            
        



KeyboardInterrupt: 